In [ ]:
import pandas as pd
import numpy as np
import chardet
import ast
from sklearn.preprocessing import MultiLabelBinarizer

with open("../dataset/portfolio.csv", "rb") as f:
    print(chardet.detect(f.read(10000)))

# 데이터 로드
portfolio = pd.read_csv("../dataset/portfolio.csv", index_col=0)
profile = pd.read_csv("../dataset/profile.csv", index_col=0)
transcript = pd.read_csv("../dataset/transcript.csv", index_col=0)
menu = pd.read_csv("../dataset/starbucks_menu_260112.csv", index_col=0)

# 데이터 형태
print("portfolio:", portfolio.shape)
print("profile:", profile.shape)
print("menu:", menu.shape)
print("transcript:", transcript.shape)

{'encoding': 'ascii', 'confidence': 1.0, 'language': 'lt'}
portfolio: (10, 6)
profile: (17000, 5)
menu: (195, 12)
transcript: (306534, 4)


In [31]:
# 데이터 타입
print("portfolio:")
print(portfolio.dtypes)

print("\nprofile:")
print(profile.dtypes)

print("menu:")
print(menu.dtypes)

print("\ntranscript:")
print(transcript.dtypes)

portfolio:
reward        int64
channels        str
difficulty    int64
duration      int64
offer_type      str
id              str
dtype: object

profile:
gender                  str
age                   int64
id                      str
became_member_on      int64
income              float64
dtype: object
menu:
제품코드            int64
제품명               str
1회 제공량(kcal)      str
포화지방(g)           str
단백질(g)            str
지방(g)             str
트랜스지방(g)          str
나트륨(mg)           str
당류(g)             str
카페인(mg)           str
콜레스테롤(mg)         str
탄수화물(g)           str
dtype: object

transcript:
person      str
event       str
value       str
time      int64
dtype: object


In [32]:
portfolio.head(10)

,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7
5,3,"['web', 'email', 'mobile', 'social']",7,7,discount,2298d6c36e964ae4a3e7e9706d1fb8c2
6,2,"['web', 'email', 'mobile', 'social']",10,10,discount,fafdcd668e3743c1bb461111dcafc2a4
7,0,"['email', 'mobile', 'social']",0,3,informational,5a8bc65990b245e5a138643cd4eb9837
8,5,"['web', 'email', 'mobile', 'social']",5,5,bogo,f19421c1d4aa40978ebb69ca19b0e20d
9,2,"['web', 'email', 'mobile']",10,7,discount,2906b810c7d4411798c6938adc9daaa5


In [33]:
profile.head(10)

,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN
5,M,68,e2127556f4f64592b11af22de27a7932,20180426,70000.0
6,NaN,118,8ec6ce2a7e7949b1bf142def7d0e0586,20170925,NaN
7,NaN,118,68617ca6246f4fbc85e91a2a49552598,20171002,NaN
8,M,65,389bc3fa690240e798340f5a15918d5c,20180209,53000.0
9,NaN,118,8974fc5686fe429db53ddde067b88302,20161122,NaN


In [34]:
menu.head(10)

,제품코드,제품명,1회 제공량(kcal),포화지방(g),단백질(g),지방(g),트랜스지방(g),나트륨(mg),당류(g),카페인(mg),콜레스테롤(mg),탄수화물(g)
0,9200000002487,나이트로 바닐라 크림,80,2,1,2.7,0,40,10,232,5,10
1,9200000000479,나이트로 콜드 브루,5,0,0,0,0,5,0,245,0,0
2,9200000002081,돌체 콜드 브루,220,6,6,10,0,80,22,155,20,24
3,9200000002407,리저브 나이트로,5,0,0,0,0,0,0,190,0,0
4,9200000002093,리저브 콜드 브루,5,0,0,0,0,0,0,190,0,0
5,9200000005282,막걸리향 크림 콜드 브루,300,13,3,16,0,65,32,53,10,35
6,9200000004312,민트 콜드 브루,100,0,0,0,0,0,23,415,0,24
7,9200000000487,바닐라 크림 콜드 브루,125,6,3,8,0,58,11,155,10,11
8,9200000005991,베르가못 콜드 브루,125,1.7,3,2.9,0,45,22,90,10,22
9,9200000006301,블랙&화이트 콜드 브루,175,12,1,14,0,70,12,190,5,12


In [35]:
transcript.head(10)

,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0
5,389bc3fa690240e798340f5a15918d5c,offer received,{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},0
6,c4863c7985cf408faee930f111475da3,offer received,{'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},0
7,2eeac8d8feae4a8cad5a6af0499a211d,offer received,{'offer id': '3f207df678b143eea3cee63160fa8bed'},0
8,aa4862eba776480b8bb9c68455b8c2e1,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
9,31dda685af34476cad5bc968bdb01c53,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0


In [36]:
portfolio.dtypes

reward        int64
channels        str
difficulty    int64
duration      int64
offer_type      str
id              str
dtype: object

# portfolio

In [37]:
portfolio = portfolio.drop(columns=['Unnamed: 0'], errors='ignore')

# 범주형으로
portfolio['offer_type'] = portfolio['offer_type'].astype('category')

# 문자열을 리스트로 변환
portfolio['channels'] = portfolio['channels'].apply(ast.literal_eval)

# 채널별 컬럼 생성 (있으면 1, 없으면 0)
portfolio['web']    = portfolio['channels'].apply(lambda x: int('web' in x))
portfolio['email']  = portfolio['channels'].apply(lambda x: int('email' in x))
portfolio['mobile'] = portfolio['channels'].apply(lambda x: int('mobile' in x))
portfolio['social'] = portfolio['channels'].apply(lambda x: int('social' in x))

# 채널 개수 합산
portfolio['channel_count'] = portfolio[['web','email','mobile','social']].sum(axis=1)

# 기존 channels 컬럼 제거
portfolio = portfolio.drop(columns=['channels'])

# profile

In [38]:
# 성별 결측치 처리
profile['gender'] = profile['gender'].fillna('U')

# 성별 원-핫 인코딩
profile = pd.get_dummies(profile, columns=['gender'])

# 나이 이상치 제거
profile = profile[profile['age'] < 100]

# 소득 결측치 처리(중앙값)
profile['income'] = profile['income'].fillna(profile['income'].median())

# 가입일 datetime 변환
profile['became_member_on'] = pd.to_datetime(profile['became_member_on'], format='%Y%m%d')


# transcript 

In [39]:
# value 컬럼 dict로
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# offer_id, amount, reward 추출
transcript['offer_id'] = transcript['value'].apply(lambda x: x.get('offer id') or x.get('offer_id'))
transcript['amount']   = transcript['value'].apply(lambda x: x.get('amount'))
transcript['reward']   = transcript['value'].apply(lambda x: x.get('reward'))

# event를 범주형으로
transcript['event'] = transcript['event'].astype('category')

# time → day 단위로
transcript['day'] = transcript['time'] // 24

# 필요 없는 value 컬럼 제거
transcript = transcript.drop(columns=['value'])

# merge

In [40]:
# transcript + portfolio
df = transcript.merge(portfolio, left_on='offer_id', right_on='id', how='left')

# transcript + profile
df = df.merge(profile, left_on='person', right_on='id', how='left')

# 불필요한 id 컬럼 제거
df = df.drop(columns=['id_x','id_y'], errors='ignore')

print(df.head())


                             person           event  time  \
0  78afa995795e4d85b5d9ceeca43f5fef  offer received     0   
1  a03223e636434f42ac4c3df47e8bac43  offer received     0   
2  e2127556f4f64592b11af22de27a7932  offer received     0   
3  8ec6ce2a7e7949b1bf142def7d0e0586  offer received     0   
4  68617ca6246f4fbc85e91a2a49552598  offer received     0   

                           offer_id  amount  reward_x  day  reward_y  \
0  9b98b8c7a33c4b65b9aebfe6a799e6d9     NaN       NaN    0       5.0   
1  0b1e1539f2cc45b7b9fa7c272da2e1d7     NaN       NaN    0       5.0   
2  2906b810c7d4411798c6938adc9daaa5     NaN       NaN    0       2.0   
3  fafdcd668e3743c1bb461111dcafc2a4     NaN       NaN    0       2.0   
4  4d5c57ea9a6940dd891ad53e9dbe8da0     NaN       NaN    0      10.0   

   difficulty  duration  ... mobile  social  channel_count   age  \
0         5.0       7.0  ...    1.0     0.0            3.0  75.0   
1        20.0      10.0  ...    0.0     0.0            2.0   NaN